In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Výchozí nastavení transpilace a možnosti konfigurace


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Abstraktní Circuit je nutné transpilovat, protože QPU mají omezený soubor základních Gate a nemohou provádět libovolné operace. Funkce Transpileru spočívá v úpravě libovolných Circuit tak, aby mohly běžet na konkrétním QPU. Toho se dosahuje překladem Circuit do podporovaných základních Gate a vložením SWAP Gate podle potřeby, aby propojení Circuit odpovídalo propojení QPU.

Jak je vysvětleno v článku [Transpilace s pass managery](transpile-with-pass-managers), můžeš vytvořit [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) pomocí funkce [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) a předat Circuit nebo seznam Circuit jeho metodě [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) k transpilaci. Funkci `generate_preset_pass_manager` můžeš zavolat s pouhým uvedením úrovně optimalizace a Backend a ponechat výchozí hodnoty pro všechny ostatní možnosti, nebo jí předat další argumenty pro jemnější doladění transpilace.

## Základní použití bez parametrů
V tomto příkladu předáme Circuit a cílové QPU Transpileru bez zadání jakýchkoli dalších parametrů.

Vytvoř Circuit a zobraz výsledek:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpiluj Circuit a zobraz výsledek:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Všechny dostupné parametry
Níže jsou uvedeny všechny dostupné parametry funkce [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Existují dvě třídy argumentů: ty, které popisují cíl kompilace, a ty, které ovlivňují způsob práce Transpileru.

Všechny parametry kromě `optimization_level` jsou volitelné. Úplné podrobnosti najdeš v [dokumentaci Transpiler API](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Míra optimalizace prováděné na Circuit. Celé číslo v rozsahu (0–3). Vyšší úrovně generují optimalizovanější Circuit za cenu delší doby transpilace. Více podrobností najdeš v článku [Nastavení úrovně optimalizace Transpileru](set-optimization).

### Parametry popisující cíl kompilace
Tyto argumenty popisují cílové QPU pro spuštění Circuit, včetně informací jako je mapa propojení QPU (popisující konektivitu Qubitů), základní Gate podporované QPU a míry chyb Gate.

Mnohé z těchto parametrů jsou podrobně popsány v článku [Běžně používané parametry pro transpilaci](common-parameters).

<details>
  <summary>
    **Parametry QPU (`Backend`)**
  </summary>

**Parametry Backend** – Pokud zadáš `backend`, nemusíš zadávat `target` ani žádné jiné možnosti Backend. Stejně tak pokud zadáš `target`, nemusíš zadávat `backend` ani žádné jiné možnosti Backend.
- `backend` (Backend) - Pokud je tato hodnota nastavena, Transpiler zkompiluje vstupní Circuit pro toto zařízení. Pokud je nastavena jiná možnost ovlivňující tato nastavení, například `coupling_map`, přepíše nastavení z `backend`.
- `target` (Target) - Cíl Transpileru Backend. Obvykle se zadává jako součást argumentu backend, ale pokud jsi ručně sestavil/a objekt Target, můžeš ho zadat zde. Přepíše cíl z `backend`.
- `backend_properties` (BackendProperties) - Vlastnosti vrácené QPU, včetně informací o chybách Gate, chybách čtení, dobách koherence Qubitů apod. QPU, které tyto informace poskytuje, najdeš spuštěním `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Volitelné omezení řídicího hardwaru na rozlišení časování instrukcí. Tyto informace jsou poskytovány konfigurací QPU. Pokud QPU nemá žádné omezení na přidělování časů instrukcí, je `timing_constraints` `None` a neprovádí se žádná úprava. QPU může nahlásit sadu omezení, konkrétně:
    - `granularity`: Celá hodnota představující minimální rozlišení pulzní Gate v jednotkách dt. Uživatelsky definovaná pulzní Gate by měla mít dobu trvání, která je násobkem této hodnoty granularity.
    - `min_length`: Celá hodnota představující minimální délku pulzní Gate v jednotkách dt. Uživatelsky definovaná pulzní Gate by měla být delší než tato délka.
    - `pulse_alignment`: Celá hodnota představující časové rozlišení počátečního času Gate instrukce. Gate instrukce by měly začínat v čase, který je násobkem této hodnoty.
    - `acquire_alignment`: Celá hodnota představující časové rozlišení počátečního času instrukce měření. Instrukce měření by měla začínat v čase, který je násobkem této hodnoty.
</details>

<details>
  <summary>
    **Parametry rozvržení a topologie**
  </summary>

- `basis_gates` (List[str] | None) - Seznam názvů základních Gate, do nichž se má rozvinout. Například ['u1', 'u2', 'u3', 'cx']. Pokud je `None`, rozvinování se neprovádí.
- `coupling_map` (CouplingMap | List[List[int]]) - Řízená mapa propojení (případně vlastní) jako cíl při mapování. Pokud je mapa propojení symetrická, je třeba zadat obě směry. Jsou podporovány tyto formáty:
    - Instance CouplingMap
    - List – musí být zadán jako matice sousednosti, kde každý záznam specifikuje všechny podporované řízené dvouqubitové interakce QPU. Například: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapování operací Circuit na pulzní harmonogramy. Pokud je `None`, použije se `instruction_schedule_map` QPU.
</details>

### Parametry ovlivňující způsob práce Transpileru
Tyto parametry mají vliv na konkrétní fáze transpilace. Některé z nich mohou ovlivňovat více fází, ale pro jednoduchost jsou uvedeny pouze pod jednou fází. Pokud zadáš argument, například `initial_layout` pro Qubit, které chceš použít, tato hodnota přepíše všechny průchody, které by ji mohly změnit. Jinými slovy, Transpiler nezmění nic, co ručně zadáš. Podrobnosti o konkrétních fázích najdeš v článku [Fáze Transpileru](transpiler-stages).

<details>
  <summary>
    **Inicializační fáze**
  </summary>

- `hls_config` (HLSConfig) - Volitelná konfigurační třída `HLSConfig`, která je předána přímo transformačnímu průchodu `HighLevelSynthesis`. Tato konfigurační třída ti umožňuje zadat seznamy syntézních algoritmů a jejich parametry pro různé vysokoúrovňové objekty.
- `init_method` (str) - Název pluginu pro inicializační fázi. Ve výchozím nastavení se nepoužívá žádný externí plugin. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `init` pro název fáze.
- `unitary_synthesis_method` (str) - Název metody unitární syntézy, která se má použít. Ve výchozím nastavení se používá `default`. Seznam nainstalovaných pluginů zobrazíš spuštěním `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Volitelný konfigurační slovník, který je předán přímo pluginu unitární syntézy. Ve výchozím nastavení toto nastavení nemá žádný účinek, protože výchozí metoda unitární syntézy nepřijímá vlastní konfiguraci. Použití vlastní konfigurace je nutné pouze tehdy, když je s argumentem `unitary_synthesis` zadán plugin unitární syntézy. Protože je toto nastavení specifické pro každý plugin unitární syntézy, přečti si dokumentaci daného pluginu, jak tuto možnost použít.
</details>

<details>
  <summary>
    **Fáze rozvržení**
  </summary>

- `initial_layout` (Layout | Dict | List) - Počáteční umístění virtuálních Qubitů na fyzické Qubit. Pokud toto rozvržení učiní Circuit kompatibilní s omezeními `coupling_map`, bude použito. Výsledné rozvržení nemusí být stejné, protože Transpiler může permutovat Qubit pomocí SWAP nebo jiných prostředků. Úplné podrobnosti najdeš v [sekci Počáteční rozvržení](common-parameters#initial-layout).
- `layout_method` (str) - Název průchodu pro výběr rozvržení (`default`, `dense`, `sabre` a `trivial`). Může to být také název externího pluginu pro fázi rozvržení. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `layout` pro `stage_name`. Výchozí hodnota je `sabre`.
</details>

<details>
  <summary>
    **Fáze směrování**
  </summary>

- `routing_method` (str) - Název průchodu pro směrování (`basic`, `lookahead`, `default`, `sabre` nebo `none`). Může to být také název externího pluginu pro fázi směrování. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `routing` pro `stage_name`. Výchozí hodnota je `sabre`.
</details>

<details>
  <summary>
    **Fáze překladu**
  </summary>

- `translation_method` (str) - Název průchodu pro překlad (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Může to být také název externího pluginu pro fázi překladu. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `translation` pro `stage_name`. Výchozí hodnota je `translator`.
</details>

<details>
  <summary>
    **Optimalizační fáze**
  </summary>

- `approximation_degree` (float, v rozsahu 0–1 | None) - Heuristický dial pro aproximaci Circuit (1.0 = žádná aproximace, 0.0 = maximální aproximace). Výchozí hodnota je 1.0. Zadání `None` nastaví stupeň aproximace na nahlášenou míru chyb. Více podrobností najdeš v [sekci Stupeň aproximace](common-parameters#approx-degree).
- `optimization_method` (str) - Název pluginu pro optimalizační fázi. Ve výchozím nastavení se nepoužívá žádný externí plugin. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `optimization` pro `stage_name`.
</details>

<details>
  <summary>
    **Fáze plánování**
  </summary>

- `scheduling_method` (str) - Název průchodu pro plánování. Může to být také název externího pluginu pro fázi plánování. Seznam nainstalovaných pluginů zobrazíš spuštěním `list_stage_plugins()` s argumentem `scheduling` pro `stage_name`.
  * 'as_soon_as_possible': Plánuje instrukce chamtivě, co nejdříve na dostupném prostředku Qubit (alias: `asap`).
  * 'as_late_as_possible': Plánuje instrukce co nejpozději, tj. udržuje Qubit v základním stavu, kdykoli je to možné (alias: `alap`). Toto je výchozí nastavení.
</details>

<details>
  <summary>
    **Spuštění Transpileru**
  </summary>

- `seed_transpiler` (int) - Nastavuje náhodná semena pro stochastické části Transpileru.
</details>

Pokud nezadáš žádný z výše uvedených parametrů, použijí se následující výchozí hodnoty. Více informací najdeš na [referenční stránce API](../api/qiskit/transpiler_preset) dané metody:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>